|                |   |
:----------------|---|
| **Nombre**     | Jocelyn Jiménez Buenrostro|
| **Fecha**      | 14/05/2026  |

Realiza una regresión logística para predecir 'y' (elimina ret_next).

Calcula los siguientes métricos:

- accuracy
- precision
- recall
- ROC AUC


Encuentra una posible limitante de este modelo.

In [63]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('ret_next.csv')


Saving ret_next.csv to ret_next (1).csv


In [64]:
df.head()

,y,ret_next,momentum,bm,size,volatility,beta,volume,spread,leverage,liquidity,sector
0,1,0.041584,0.012829,-0.054778,1809.117227,0.030864,-0.480847,2297.572303,0.001586,0.499458,529.660760,NaN
1,0,-0.093096,0.479827,-0.922413,217715.092700,0.040858,-0.624887,6239.575395,0.005340,0.369037,380.973412,Energy
2,1,0.060016,-0.268646,0.007363,5488.594733,0.181708,0.366617,1835.958388,0.017419,3.207282,697.893788,Healthcare
3,0,0.025898,-1.836360,-1.478547,4249.689724,0.038738,0.205193,2781.488057,0.005470,0.257056,105.690037,Industrials
4,0,-0.046861,-0.432774,-0.487633,61240.893023,0.158916,-0.606618,5112.926784,0.001779,0.805322,192.022707,Technology


In [65]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   y           1500 non-null   int64  
 1   ret_next    1500 non-null   float64
 2   momentum    1426 non-null   float64
 3   bm          1442 non-null   float64
 4   size        1500 non-null   float64
 5   volatility  1454 non-null   float64
 6   beta        1500 non-null   float64
 7   volume      1500 non-null   float64
 8   spread      1500 non-null   float64
 9   leverage    1500 non-null   float64
 10  liquidity   1500 non-null   float64
 11  sector      1472 non-null   object 
dtypes: float64(10), int64(1), object(1)
memory usage: 140.8+ KB


In [66]:
df.describe()

,y,ret_next,momentum,bm,size,volatility,beta,volume,spread,leverage,liquidity
count,1500.000000,1500.000000,1426.000000,1442.000000,1.500000e+03,1454.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000
mean,0.530667,0.000709,-0.028957,-0.045971,3.875114e+04,0.166114,0.023438,5216.713201,0.015738,1.621723,707.586596
std,0.499225,0.059198,1.073846,1.040225,6.659258e+04,0.237555,0.935255,7174.115398,0.018416,2.358823,953.746328
min,0.000000,-0.174651,-8.305417,-3.221016,1.075443e+03,0.020000,-3.211588,162.344392,0.000500,0.050000,8.700186
25%,0.000000,-0.038800,-0.677544,-0.766253,1.159125e+04,0.048965,-0.598279,1529.889467,0.005155,0.499079,204.545958
50%,1.000000,-0.000403,-0.009567,-0.051330,2.269942e+04,0.098347,0.022000,3064.361457,0.009866,0.972993,404.151156
75%,1.000000,0.041824,0.659620,0.674450,4.393697e+04,0.189748,0.654069,6004.397622,0.018999,1.861845,831.112875
max,1.000000,0.182306,7.963350,3.529055,1.816737e+06,5.071581,2.915867,79976.729134,0.212487,32.034515,11818.045466


In [67]:
df.isnull().sum()

,0
y,0
ret_next,0
momentum,74
bm,58
size,0
volatility,46
beta,0
volume,0
spread,0
leverage,0


In [68]:
df = df.dropna()

In [69]:
X = df.drop(columns=["y","ret_next"])

In [70]:
y = df["y"]

In [71]:
cat_cols = X.select_dtypes(include=["object"]).columns

num_cols = X.select_dtypes(exclude=["object"]).columns

In [72]:
print("Variables categóricas:")
print(cat_cols)

print("\nVariables numéricas:")
print(num_cols)

Variables categóricas:
Index(['sector'], dtype='object')

Variables numéricas:
Index(['momentum', 'bm', 'size', 'volatility', 'beta', 'volume', 'spread',
       'leverage', 'liquidity'],
      dtype='object')


In [73]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

In [74]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        random_state=42
    ))
])

In [75]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.20,
    random_state=1

)

In [76]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['momentum', 'bm', 'size', 'volatility', 'beta', 'volume', 'spread',
       'leverage', 'liquidity'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['sector'], dtype='object'))])),
                ('model', LogisticRegression(random_state=42))])

In [77]:
from sklearn.metrics import (accuracy_score,roc_auc_score,precision_score,recall_score)

In [78]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:,1]


accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("ROC AUC:", roc_auc)

Accuracy: 0.7203065134099617
Precision: 0.7315436241610739
Recall: 0.7676056338028169
ROC AUC: 0.8069594034797017


2. Realiza una regresión lineal sobre ret_next (no uses 'y'), e identifica si se cuenta con alguno de los 6 problemas comunes.

¿Cuáles problemas encontraste?

¿Cómo los arreglaste?

In [79]:
X = pd.get_dummies(X, drop_first=True)

In [80]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.20,
    random_state=1

)

IndentationError: unexpected indent (3721532726.py, line 5)

In [ ]:
cat_cols = X.select_dtypes(include=["object"]).columns

num_cols = X.select_dtypes(exclude=["object"]).columns

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

In [ ]:
from sklearn.linear_model import LinearRegression
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:,1]


accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("ROC AUC:", roc_auc)

3. Utiliza el dataset corregido (producto del punto 2) para entrenar un árbol de decisión para clasificación. Optimiza los hiperparámetros max_depth & min_samples_leaf. Compara los métricos del árbol con los medidos en el punto 1.

